# BP3 Gate 2 — Data Verification & Feature/Taxonomy Engineering
**Customer360 Navigator Enterprise Suite — Complaint Escalation / Intervention Prediction**

## Why this notebook exists, and how it differs from BP1's and BP2's Gate 2
BP3 Gate 1's real run (2026-09-23, `Company response to consumer` distribution confirmed:
Closed with explanation 492,340 / Closed with non-monetary relief 312,602 / In progress 231,856 /
Closed with monetary relief 10,511 / Untimely response 1,264 / null 2) already fully and
unambiguously defined `intervention_required` directly from that one real field's literal values —
unlike BP1 Gate 2 (which had to build a genuinely new CFPB<->BANKING77 crosswalk) or BP2 Gate 2
(which had to build a genuinely new severity bucket-to-class judgment mapping), BP3 has no
taxonomy left to invent, and BP3 does not integrate BANKING77 at all (Master Plan BP table:
`Integrates BANKING77? = NO`). So this notebook's real work is the two things Gate 2's own exit
criteria (Master Plan Section 8) actually require here: **zero nulls silently dropped** in the
real candidate feature columns, and a **complete feature-lineage table** — plus tagging every real
row with its target/exclusion status and writing the Gold layer, the same "every row tagged, none
dropped" pattern BP2 Gate 2 used.

## What this notebook does
1. Re-verifies live (zero-fabrication) that `Company response to consumer` still matches the
   distribution BP3 Gate 1's `policy.json` recorded — never trusts the baked-in counts without
   re-measuring.
2. Live-screens the 7 real candidate feature columns (`Product`, `Sub-product`, `Issue`,
   `Sub-issue`, `State`, `Submitted via`, `Company`) for nulls, and compares the live counts against
   `DATA_PROFILE_REPORT.md`'s documented counts — a second independent drift check, on columns Gate
   1 never examined (Gate 1 only checked the target/leakage-rule columns).
3. Applies an **explicit, named sentinel fill** to every candidate column with a real null count
   (`Sub-product`, `Sub-issue`, `State`) via the new `src/features/bp3_escalation_features.py`
   module — never a silent drop, satisfying Gate 2's own exit criterion by construction, not by
   assertion after the fact.
4. Tags every real row with `intervention_required` / `exclusion_reason` using that same module's
   `intervention_target_expr()`, and cross-checks the live result against Gate 1's `policy.json`
   `target_class_balance` — this must match exactly, since both are the identical rule applied to
   the identical file; any mismatch is a real bug, not measurement noise.
5. Builds and writes a **feature-lineage table** (`feature_lineage_table()`) — one row per
   engineered feature, its real source column, its transform, and its null-handling — the artifact
   Gate 2's exit criterion names explicitly.
6. Writes the BP3 CFPB-with-target Gold layer to Parquet (WARP: Parquet over CSV for reused data)
   — every real row tagged, none dropped here; filtering to the trainable subset is a Gate 3
   decision, recorded here only as a real count.
7. Re-confirms the PII/compliance touchpoint (Master Plan Section 8, Gate 2 row: "PII screen on
   narrative text before any downstream/external call") is structurally satisfied by there being no
   narrative-text column in this real 15-column CFPB extract at all — checked live against the
   actual scanned column list, not assumed from RAW_DATA_MANIFEST.md's earlier finding.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; it does not run it. You run it on your own
  machine, and the real, live-checked results below become this project's Gate 2 record for BP3.
- **Zero-fabrication**: every null count, row-accounting number, and distribution below is
  computed live against the real CFPB file or the real Gate 1 `policy.json` artifact — nothing is
  asserted from memory or from this notebook's own markdown.
- **WARP**: `configure_performance()` first. Reads the same real 1,048,575-row CFPB file via
  `pl.scan_csv` (lazy), exactly as Gate 1 does.
- **HYPER**: this notebook is the first to import `src/features/bp3_escalation_features.py`
  (new — built now, at Gate 2, rather than triplicated across Gates 3/4/5 first and centralized
  only at Gate 6 the way BP2's `bp2_friction_features.py` had to be; see that module's own
  docstring for the technical debt this avoids). Also reuses `taxonomy.taxonomy_mapper.CFPB_DTYPES`
  (no re-derivation) and `src/utils/bp1_config_sync.py` (fourth BP to reuse it unmodified).
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.
- **Idempotent**: re-running this notebook overwrites the Gold Parquet, the feature-lineage CSV,
  and this config's own `gate2` block in place.

## Outputs (idempotent overwrite-in-place)
- `data/processed/cfpb_intervention_escalation_gold.parquet` — every real row, tagged
- `notebooks/bp3_complaint_escalation_prediction/artifacts/gate2_feature_lineage.csv`
- `configs/bp3_complaint_escalation_prediction.yaml` — new `gate2` block appended (front matter
  and any other existing gate blocks preserved verbatim via `write_gate_block`)

## Prerequisites
BP3 Gate 1 must have run for real at least once (`configs/bp3_complaint_escalation_prediction.yaml`
front matter and `notebooks/bp3_complaint_escalation_prediction/artifacts/policy.json` must both
exist) — this notebook reads Gate 1's real recorded `target_class_balance` to cross-check its own
live target-tagging result, rather than assuming Gate 1 to have been correct without re-verifying.

## If a structural check below fails
It raises `AssertionError` with the failing check named. A mismatch between this notebook's live
target tagging and Gate 1's recorded `target_class_balance` in particular must never be worked
around silently — it means the two notebooks' target rules have drifted apart, which is exactly
the kind of inconsistency Gate 2's re-verification exists to catch.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP3 Gate 2 data verification / feature engineering
notebook. Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb

perf_summary = configure_performance(project_root=PROJECT_ROOT)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import polars as pl
from IPython.display import display

from taxonomy.taxonomy_mapper import CFPB_DTYPES
from features.bp3_escalation_features import (
    BARRED_COLUMNS,
    null_screen_report,
    load_cfpb_with_target_and_filled_features,
    feature_lineage_table,
    build_bp3_escalation_gold_layer,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

CFPB_PATH = DATA_RAW / "cfpb_complaints.csv"
BP3_CONFIG_PATH = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"
POLICY_JSON_PATH = ARTIFACTS_DIR / "policy.json"

for p in (CFPB_PATH, BP3_CONFIG_PATH, POLICY_JSON_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP3 Gate 1 has been real-run at least once "
            "(the config front matter and policy.json artifact are both prerequisites)."
        )

with open(POLICY_JSON_PATH, "r", encoding="utf-8") as f:
    gate1_policy = json.load(f)
print(f"[OK] Loaded Gate 1 policy.json (generated_at_utc={gate1_policy['generated_at_utc']}).")

# ============================================================
# SECTION 4: LIVE drift check - re-measure 'Company response to consumer' against what Gate 1's
# policy.json recorded. Zero-fabrication: never trust the artifact's baked-in counts without
# re-measuring against the real file.
# ============================================================
live_company_response_counts = (
    pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
    .group_by("Company response to consumer")
    .agg(pl.len().alias("row_count"))
    .collect()
)
live_company_response_dict = dict(
    zip(
        live_company_response_counts["Company response to consumer"].cast(pl.Utf8).to_list(),
        live_company_response_counts["row_count"].to_list(),
    )
)
documented_company_response_dict = {
    d["Company response to consumer"]: d["n"]
    for d in gate1_policy["live_checks"]["company_response_to_consumer_distribution"]
}

company_response_drift = {
    k: {"documented": documented_company_response_dict.get(k), "live": live_company_response_dict.get(k)}
    for k in set(documented_company_response_dict) | set(live_company_response_dict)
    if documented_company_response_dict.get(k) != live_company_response_dict.get(k)
}

if company_response_drift:
    print(f"[DRIFT DETECTED] company_response_drift={json.dumps(company_response_drift, indent=2)}")
else:
    print(
        "[OK] Live 'Company response to consumer' distribution matches Gate 1's policy.json "
        "exactly - no drift since BP3 Gate 1's real run."
    )

# ============================================================
# SECTION 5: LIVE null screen on the 7 real candidate feature columns - compared against
# DATA_PROFILE_REPORT.md's documented counts. Gate 2's own exit criterion: zero nulls silently
# dropped - this section measures the real numbers that criterion is checked against.
# ============================================================
cfpb_lazy_base = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)
live_null_report = null_screen_report(cfpb_lazy_base)
print("\n=== LIVE NULL SCREEN: BP3 candidate feature columns ===")
display(live_null_report.to_pandas())

documented_null_counts = {
    "Product": 0,
    "Sub-product": 21,
    "Issue": 0,
    "Sub-issue": 25808,
    "State": 2193,
    "Submitted via": 0,
    "Company": 0,
}
null_drift = {
    row["column"]: {"documented": documented_null_counts.get(row["column"]), "live": row["null_count"]}
    for row in live_null_report.to_dicts()
    if documented_null_counts.get(row["column"]) != row["null_count"]
}
if null_drift:
    print(f"[DRIFT DETECTED] null_count_drift={json.dumps(null_drift, indent=2)}")
else:
    print(
        "[OK] Live null counts for every BP3 candidate feature column match "
        "DATA_PROFILE_REPORT.md exactly - no drift."
    )

# ============================================================
# SECTION 6: Live column-list check for the PII/narrative-text compliance touchpoint (Master Plan
# Section 8, Gate 2 row) - confirms live, not assumed, that no narrative-text column exists in
# this real extract, so no PII screening step is being silently skipped.
# ============================================================
live_columns = cfpb_lazy_base.collect_schema().names()
NARRATIVE_TEXT_KEYWORDS = ("narrative", "complaint text", "free text", "consumer complaint narrative")
narrative_columns_found = [c for c in live_columns if any(kw in c.lower() for kw in NARRATIVE_TEXT_KEYWORDS)]
print(f"\n[COMPLIANCE CHECK] Real CFPB columns (live): {live_columns}")
if narrative_columns_found:
    print(
        f"[ACTION REQUIRED] Narrative-text column(s) found: {narrative_columns_found} - a PII "
        "screen is required before this data can be used downstream. NOT performed by this "
        "notebook."
    )
else:
    print(
        "[OK] No narrative-text column present in this real 15-column CFPB extract (live-"
        "confirmed) - the PII-screen-on-narrative-text compliance touchpoint has no narrative "
        "text to screen for BP3, mirroring RAW_DATA_MANIFEST.md's earlier finding."
    )

# ============================================================
# SECTION 7: Tag every real row with intervention_required/exclusion_reason, apply the explicit
# null-sentinel fill, and cross-check the live result against Gate 1's recorded target_class_balance.
# ============================================================
cfpb_lazy_tagged = load_cfpb_with_target_and_filled_features(CFPB_PATH)

live_target_balance = (
    cfpb_lazy_tagged.group_by(["intervention_required", "exclusion_reason"])
    .agg(pl.len().alias("row_count"))
    .collect()
    .sort("row_count", descending=True)
)
print("\n=== LIVE TARGET TAGGING: intervention_required x exclusion_reason ===")
display(live_target_balance.to_pandas())

n_intervention_required = int(
    live_target_balance.filter(pl.col("intervention_required") == 1)["row_count"].sum()
)
n_no_intervention_required = int(
    live_target_balance.filter(pl.col("intervention_required") == 0)["row_count"].sum()
)
n_excluded_in_progress = int(
    live_target_balance.filter(pl.col("exclusion_reason") == "EXCLUDED_PENDING")["row_count"].sum()
)
n_excluded_untimely_response = int(
    live_target_balance.filter(pl.col("exclusion_reason") == "EXCLUDED_UNTIMELY_RESPONSE_OVERLAPS_BP2")[
        "row_count"
    ].sum()
)
n_excluded_null_response = int(
    live_target_balance.filter(pl.col("exclusion_reason") == "EXCLUDED_UNKNOWN_NULL_RESPONSE")[
        "row_count"
    ].sum()
)
n_trainable_total = n_intervention_required + n_no_intervention_required
n_excluded_total = n_excluded_in_progress + n_excluded_untimely_response + n_excluded_null_response

gate1_balance = gate1_policy["live_checks"]["target_class_balance"]
target_balance_matches_gate1 = (
    n_intervention_required == gate1_balance["n_intervention_required"]
    and n_no_intervention_required == gate1_balance["n_no_intervention_required"]
    and n_excluded_in_progress == gate1_balance["n_excluded_in_progress"]
    and n_excluded_untimely_response == gate1_balance["n_excluded_untimely_response"]
    and n_excluded_null_response == gate1_balance["n_excluded_null_response"]
)
print(
    f"\n[FINDING] Live target tagging: {n_intervention_required:,} intervention_required=1, "
    f"{n_no_intervention_required:,} intervention_required=0, {n_trainable_total:,} trainable "
    f"total, {n_excluded_total:,} excluded total. Matches Gate 1's recorded "
    f"target_class_balance exactly: {target_balance_matches_gate1}"
)

# ============================================================
# SECTION 8: Build and save the feature-lineage table (Gate 2's own named exit-criterion artifact)
# ============================================================
lineage = feature_lineage_table()
print("\n=== FEATURE LINEAGE TABLE ===")
display(lineage.to_pandas())

lineage_path = ARTIFACTS_DIR / "gate2_feature_lineage.csv"
lineage.write_csv(lineage_path)
print(f"[SAVED] {lineage_path.relative_to(PROJECT_ROOT)}")

# A barred column is allowed to appear in the lineage table as EITHER "(none - barred)" (the
# explicit exclusion row) OR the target row itself (source_column == the field that defines
# intervention_required - it is the LABEL, never an input feature, tagged "TARGET" in its own
# engineered_feature string precisely so this check can tell the two legitimate cases apart from
# a real leak).
barred_in_lineage_as_feature = lineage.filter(
    pl.col("source_column").is_in(BARRED_COLUMNS)
    & (pl.col("engineered_feature") != "(none - barred)")
    & (~pl.col("engineered_feature").str.contains("TARGET"))
)
no_barred_column_used_as_feature = barred_in_lineage_as_feature.height == 0

# ============================================================
# SECTION 9: Write the BP3 CFPB-with-target Gold layer (Parquet, WARP)
# ============================================================
gold_summary = build_bp3_escalation_gold_layer(cfpb_lazy_tagged, DATA_PROCESSED)
print(
    f"\n[SAVED] BP3 CFPB Escalation Gold: {gold_summary['cfpb_escalation_gold_path']} "
    f"({gold_summary['cfpb_escalation_gold_rows_written']} rows)"
)

# ============================================================
# SECTION 10: Write the Gate 2 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, fourth BP to do so)
# ============================================================
from utils.bp1_config_sync import write_gate_block  # noqa: E402

gate2_marker = (
    "# --- Gate 2 (Data Verification & Feature/Taxonomy Engineering) results "
    "(appended, idempotent overwrite) ---"
)
gate2_block_lines = (
    [
        f"drift_vs_gate1_live_check: {'none' if not company_response_drift else 'DRIFT_DETECTED'}",
        f"null_count_drift_vs_data_profile_report: {'none' if not null_drift else 'DRIFT_DETECTED'}",
        f"narrative_text_column_found: {bool(narrative_columns_found)}",
        "candidate_feature_null_counts:",
    ]
    + [f"  {row['column']}: {row['null_count']}" for row in live_null_report.to_dicts()]
    + [
        f"n_intervention_required: {n_intervention_required}",
        f"n_no_intervention_required: {n_no_intervention_required}",
        f"n_trainable_total: {n_trainable_total}",
        f"n_excluded_in_progress: {n_excluded_in_progress}",
        f"n_excluded_untimely_response: {n_excluded_untimely_response}",
        f"n_excluded_null_response: {n_excluded_null_response}",
        f"n_excluded_total: {n_excluded_total}",
        f"target_balance_matches_gate1_policy_json: {target_balance_matches_gate1}",
        f'feature_lineage_path: "{lineage_path.relative_to(PROJECT_ROOT).as_posix()}"',
        f"no_barred_column_used_as_feature: {no_barred_column_used_as_feature}",
        'cfpb_escalation_gold_path: "'
        + Path(gold_summary["cfpb_escalation_gold_path"]).relative_to(PROJECT_ROOT).as_posix()
        + '"',
        f"cfpb_escalation_gold_rows_written: {gold_summary['cfpb_escalation_gold_rows_written']}",
    ]
)
write_gate_block(BP3_CONFIG_PATH, gate2_marker, gate2_block_lines)
print(f"[SAVED] gate2 block written to {BP3_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "no_drift_vs_gate1_company_response_distribution": not company_response_drift,
    "no_null_count_drift_vs_data_profile_report": not null_drift,
    "no_narrative_text_column_present": not narrative_columns_found,
    "target_tagging_matches_gate1_policy_json": target_balance_matches_gate1,
    "trainable_plus_excluded_equals_total": (
        n_trainable_total + n_excluded_total == gate1_policy["live_checks"]["cfpb_row_count"]
    ),
    "feature_lineage_csv_written": lineage_path.exists(),
    "no_barred_column_used_as_engineered_feature": no_barred_column_used_as_feature,
    "gold_layer_row_count_matches_source": (
        gold_summary["cfpb_escalation_gold_rows_written"] == gate1_policy["live_checks"]["cfpb_row_count"]
    ),
    "bp3_config_gate2_block_written": BP3_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP3 Gate 2 complete - every real row tagged with "
    f"intervention_required/exclusion_reason ({n_trainable_total:,} trainable, "
    f"{n_excluded_total:,} excluded, none dropped), zero nulls silently dropped in any "
    "candidate feature column, feature-lineage table written. Proceed to BP3 Gate 3 "
    "(Model/Classifier Benchmark & Champion Selection) next."
)
